# Dream-RSI: Recursive Self-Improvement via Tree of Thoughts (7B Model)

This notebook demonstrates **RLT (Reinforcement Learning Tree) + RSI (Recursive Self-Improvement)** on a 7B/8B parameter LLM (e.g., `qwen2.5-coder:7b`). 

Instead of just asking the model to write code once, we use an **orchestration layer** that allows the model to:
1. Generate multiple branching solutions (Tree exploration).
2. Evaluate those solutions against test cases.
3. Backtrack and self-correct (RSI) if a branch fails.

*Inspired by Zheng et al. (2026).* Run this on an L4 GPU.

In [ ]:
!pip install -q requests

# 1. Start Ollama Server in the background
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(3)

# 2. Pull a 7B/8B model (perfect for L4 GPU)
!ollama pull qwen2.5-coder:7b

In [ ]:
import requests
import json

def query_model(prompt: str) -> str:
    """Query the local 7B model"""
    url = 'http://localhost:11434/api/generate'
    payload = {
        'model': 'qwen2.5-coder:7b',
        'prompt': prompt,
        'stream': False,
        'options': {'temperature': 0.7} # allow exploration
    }
    response = requests.post(url, json=payload)
    return response.json()['response']

def evaluate_code(code: str, tests: list) -> float:
    """Dummy evaluator: in reality, this executes the code in a sandbox."""
    # Simulated reward based on keywords for demonstration
    score = 0.0
    if "def " in code: score += 0.2
    if "return " in code: score += 0.2
    if "for " in code or "while " in code: score += 0.4
    if "Bug" not in code: score += 0.2
    return score


In [ ]:
# --- The RLT (Tree Exploration) + RSI (Self-Correction) Loop ---

problem = "Write a Python function to sort an array using Merge Sort."
max_branches = 3
max_depth = 2

def rsi_tree_search(problem: str, depth: int = 0):
    if depth >= max_depth:
        return None, 0.0
        
    print(f"\n[Depth {depth}] Generating {max_branches} branch proposals...")
    branches = []
    
    # 1. Branching (Tree of Thoughts)
    for i in range(max_branches):
        prompt = f"{problem}\n\nAttempt {i+1}: Provide ONLY the Python code."
        code = query_model(prompt)
        score = evaluate_code(code, [])
        branches.append({'code': code, 'score': score})
        print(f"  -> Branch {i+1} Score: {score}")
        
    # 2. Evaluation
    best_branch = max(branches, key=lambda x: x['score'])
    if best_branch['score'] >= 1.0:
        print("\n[Success] Found perfect solution!")
        return best_branch['code'], best_branch['score']
        
    # 3. Self-Correction (Recursive Self-Improvement)
    print(f"\n[Depth {depth}] Best score {best_branch['score']} < 1.0. Applying self-correction...")
    feedback_prompt = f"The following code scored {best_branch['score']}/1.0. Fix the bugs and improve it.\n\nCode:\n{best_branch['code']}"
    return rsi_tree_search(feedback_prompt, depth + 1)

# Execute the loop!
final_code, final_score = rsi_tree_search(problem)
print(f"\n=== FINAL RESULT (Score: {final_score}) ===\n{final_code}")
